<a href="https://colab.research.google.com/github/LauraOminde01/plant-doctor-/blob/main/Crop_Doctor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# Write to the correct location
token = userdata.get('KAGGLE_API_TOKEN').strip()
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)
os.chmod('/root/.kaggle/access_token', 0o600)

print("Kaggle authenticated")

Kaggle authenticated


In [ ]:
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# Download PlantVillage dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset --quiet
!unzip -q new-plant-diseases-dataset.zip -d data/
print("Downloaded and extracted")

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
Downloaded and extracted


In [ ]:
import os

base = 'data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
train_path = os.path.join(base, 'train')

all_classes = os.listdir(train_path)
print(f"Total classes: {len(all_classes)}")
print("\nAll classes:")
for c in sorted(all_classes):
    print(f"  {c}")

# Filter to Kenya-relevant crops
kenya_crops = ['Corn', 'Potato', 'Tomato', 'Wheat']
kenya_classes = [c for c in all_classes if any(crop in c for crop in kenya_crops)]
print(f"\nKenya-relevant classes ({len(kenya_classes)}):")
for c in sorted(kenya_classes):
    count = len(os.listdir(os.path.join(train_path, c)))
    print(f"  {c}: {count} images")

Total classes: 38

All classes:
  Apple___Apple_scab
  Apple___Black_rot
  Apple___Cedar_apple_rust
  Apple___healthy
  Blueberry___healthy
  Cherry_(including_sour)___Powdery_mildew
  Cherry_(including_sour)___healthy
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  Corn_(maize)___Common_rust_
  Corn_(maize)___Northern_Leaf_Blight
  Corn_(maize)___healthy
  Grape___Black_rot
  Grape___Esca_(Black_Measles)
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
  Grape___healthy
  Orange___Haunglongbing_(Citrus_greening)
  Peach___Bacterial_spot
  Peach___healthy
  Pepper,_bell___Bacterial_spot
  Pepper,_bell___healthy
  Potato___Early_blight
  Potato___Late_blight
  Potato___healthy
  Raspberry___healthy
  Soybean___healthy
  Squash___Powdery_mildew
  Strawberry___Leaf_scorch
  Strawberry___healthy
  Tomato___Bacterial_spot
  Tomato___Early_blight
  Tomato___Late_blight
  Tomato___Leaf_Mold
  Tomato___Septoria_leaf_spot
  Tomato___Spider_mites Two-spotted_spider_mite
  Tomato___Target_Spot

In [ ]:
# Kenya-specific disease information
# This is the core of what makes this project unique
DISEASE_INFO = {
    # MAIZE (Mahindi)
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': {
        'english_name': 'Maize Gray Leaf Spot',
        'swahili_name': 'Ugonjwa wa Madoa ya Kijivu (Mahindi)',
        'severity': 'High',
        'description': 'Fungal disease causing rectangular gray-brown lesions on maize leaves, reducing photosynthesis and yield.',
        'treatment': [
            'Apply fungicides containing strobilurin or triazole compounds',
            'Plant resistant maize varieties (e.g., DK8031, H614D)',
            'Practice crop rotation — avoid planting maize in the same field consecutively',
            'Remove and destroy infected crop residue after harvest',
            'Ensure adequate spacing between plants for good air circulation'
        ],
        'prevention': 'Use certified disease-resistant seeds. Avoid overhead irrigation.',
        'urgency': 'Act within 1-2 weeks of first signs'
    },

    'Corn_(maize)___Common_rust_': {
        'english_name': 'Maize Common Rust',
        'swahili_name': 'Kutu ya Mahindi',
        'severity': 'Medium',
        'description': 'Fungal disease producing powdery orange-brown pustules on both sides of maize leaves.',
        'treatment': [
            'Apply fungicides early — mancozeb or copper-based fungicides are effective',
            'Plant rust-resistant hybrid varieties',
            'Early planting helps avoid peak rust season',
            'Remove severely infected plants to reduce spread'
        ],
        'prevention': 'Monitor fields regularly during wet, humid weather when rust spreads fastest.',
        'urgency': 'Act within 2-3 weeks of first signs'
    },

    'Corn_(maize)___Northern_Leaf_Blight': {
        'english_name': 'Maize Northern Leaf Blight',
        'swahili_name': 'Ugonjwa wa Majani ya Mahindi (Kaskazini)',
        'severity': 'High',
        'description': 'Fungal disease causing large cigar-shaped gray-green lesions on maize leaves, severely reducing yield.',
        'treatment': [
            'Apply fungicides containing propiconazole or azoxystrobin',
            'Plant blight-resistant hybrid varieties available from KALRO',
            'Practice crop rotation with non-maize crops for at least one season',
            'Remove and burn infected plant debris',
            'Avoid excessive nitrogen fertilizer which promotes disease'
        ],
        'prevention': 'Use certified seeds. Ensure proper drainage in fields.',
        'urgency': 'Act within 1 week of first signs — spreads rapidly'
    },

    'Corn_(maize)___healthy': {
        'english_name': 'Healthy Maize',
        'swahili_name': 'Mahindi Yenye Afya',
        'severity': 'None',
        'description': 'Your maize crop appears healthy. No disease detected.',
        'treatment': ['Continue current farming practices', 'Monitor regularly for early disease signs'],
        'prevention': 'Maintain soil health, proper spacing, and balanced fertilization.',
        'urgency': 'No action required'
    },

    # POTATO (Viazi)
    'Potato___Early_blight': {
        'english_name': 'Potato Early Blight',
        'swahili_name': 'Ugonjwa wa Mapema wa Viazi',
        'severity': 'Medium',
        'description': 'Fungal disease causing dark brown spots with yellow rings (target-board pattern) on potato leaves.',
        'treatment': [
            'Apply fungicides containing chlorothalonil or mancozeb every 7-10 days',
            'Remove and destroy infected leaves immediately',
            'Avoid overhead irrigation — water at soil level only',
            'Ensure adequate potassium fertilization to boost plant immunity',
            'Harvest promptly when mature to avoid tuber infection'
        ],
        'prevention': 'Use certified disease-free seed potatoes. Practice 2-3 year crop rotation.',
        'urgency': 'Act within 1-2 weeks of first signs'
    },

    'Potato___Late_blight': {
        'english_name': 'Potato Late Blight',
        'swahili_name': 'Ugonjwa wa Marehemu wa Viazi',
        'severity': 'Critical',
        'description': 'Highly destructive fungal disease causing water-soaked dark lesions. The disease that caused the Irish Famine. Can destroy an entire crop within days in wet conditions.',
        'treatment': [
            'URGENT: Apply fungicides immediately — metalaxyl or cymoxanil are most effective',
            'Remove and destroy all infected plants — do not compost',
            'Stop overhead irrigation completely',
            'Harvest remaining healthy tubers immediately if infection is severe',
            'Inform neighboring farmers — late blight spreads very rapidly'
        ],
        'prevention': 'Plant resistant varieties. Never use infected tubers as seed.',
        'urgency': 'IMMEDIATE ACTION REQUIRED — can destroy entire crop within 1 week'
    },

    'Potato___healthy': {
        'english_name': 'Healthy Potato',
        'swahili_name': 'Viazi Vyenye Afya',
        'severity': 'None',
        'description': 'Your potato crop appears healthy. No disease detected.',
        'treatment': ['Continue current farming practices', 'Monitor weekly for disease signs'],
        'prevention': 'Use certified seed potatoes. Practice proper crop rotation.',
        'urgency': 'No action required'
    },

    # TOMATO (Nyanya)
    'Tomato___Bacterial_spot': {
        'english_name': 'Tomato Bacterial Spot',
        'swahili_name': 'Madoa ya Bakteria ya Nyanya',
        'severity': 'High',
        'description': 'Bacterial disease causing small water-soaked spots on leaves, stems, and fruits, reducing marketability.',
        'treatment': [
            'Apply copper-based bactericides every 7 days',
            'Remove and destroy infected plant parts',
            'Avoid working in the field when plants are wet',
            'Disinfect tools between plants',
            'Improve field drainage'
        ],
        'prevention': 'Use disease-free certified seeds. Avoid overhead irrigation.',
        'urgency': 'Act within 1 week of first signs'
    },

    'Tomato___Early_blight': {
        'english_name': 'Tomato Early Blight',
        'swahili_name': 'Ugonjwa wa Mapema wa Nyanya',
        'severity': 'Medium',
        'description': 'Fungal disease causing dark brown spots with concentric rings on lower leaves first.',
        'treatment': [
            'Apply fungicides containing chlorothalonil or mancozeb',
            'Remove infected lower leaves',
            'Mulch around plants to prevent soil splash',
            'Ensure good air circulation through proper spacing',
            'Water at soil level, not overhead'
        ],
        'prevention': 'Rotate crops. Use resistant varieties when available.',
        'urgency': 'Act within 2 weeks of first signs'
    },

    'Tomato___Late_blight': {
        'english_name': 'Tomato Late Blight',
        'swahili_name': 'Ugonjwa wa Marehemu wa Nyanya',
        'severity': 'Critical',
        'description': 'Destructive disease causing dark water-soaked lesions on leaves and brown rot on fruits.',
        'treatment': [
            'URGENT: Apply metalaxyl or cymoxanil fungicides immediately',
            'Remove and destroy all infected plants',
            'Harvest any mature fruits immediately before infection spreads',
            'Avoid irrigation until disease is controlled',
            'Alert neighboring farmers immediately'
        ],
        'prevention': 'Plant resistant varieties. Never plant tomatoes near infected potatoes.',
        'urgency': 'IMMEDIATE ACTION REQUIRED'
    },

    'Tomato___Leaf_Mold': {
        'english_name': 'Tomato Leaf Mold',
        'swahili_name': 'Ukungu wa Majani ya Nyanya',
        'severity': 'Medium',
        'description': 'Fungal disease causing yellow patches on upper leaf surface and olive-green mold on underside.',
        'treatment': [
            'Improve ventilation — increase plant spacing',
            'Apply fungicides containing chlorothalonil or copper',
            'Reduce humidity in greenhouse environments',
            'Remove and destroy infected leaves'
        ],
        'prevention': 'Most common in humid, poorly ventilated greenhouses. Ensure good airflow.',
        'urgency': 'Act within 2 weeks of first signs'
    },

    'Tomato___Septoria_leaf_spot': {
        'english_name': 'Tomato Septoria Leaf Spot',
        'swahili_name': 'Madoa ya Majani ya Nyanya',
        'severity': 'Medium',
        'description': 'Fungal disease causing small circular spots with dark borders and light centers on lower leaves.',
        'treatment': [
            'Apply fungicides containing chlorothalonil or copper hydroxide',
            'Remove infected lower leaves',
            'Avoid wetting foliage when watering',
            'Mulch to prevent soil splash'
        ],
        'prevention': 'Rotate crops. Remove plant debris at end of season.',
        'urgency': 'Act within 2 weeks of first signs'
    },

    'Tomato___Spider_mites Two-spotted_spider_mite': {
        'english_name': 'Tomato Spider Mites',
        'swahili_name': 'Utitiri wa Nyanya',
        'severity': 'Medium',
        'description': 'Tiny mites (not a fungus) causing yellow stippling on leaves. Look for fine webbing under leaves.',
        'treatment': [
            'Apply miticides or insecticidal soap',
            'Spray plants with strong water stream to dislodge mites',
            'Introduce predatory mites (biological control)',
            'Avoid over-fertilizing with nitrogen which attracts mites',
            'Remove heavily infested leaves'
        ],
        'prevention': 'Spider mites thrive in hot, dry conditions. Maintain adequate soil moisture.',
        'urgency': 'Act within 1-2 weeks — populations grow very rapidly'
    },

    'Tomato___Target_Spot': {
        'english_name': 'Tomato Target Spot',
        'swahili_name': 'Madoa ya Lengo ya Nyanya',
        'severity': 'Medium',
        'description': 'Fungal disease causing circular brown spots with concentric rings resembling a target on leaves and fruit.',
        'treatment': [
            'Apply fungicides containing chlorothalonil or azoxystrobin',
            'Improve air circulation through pruning',
            'Remove infected plant material',
            'Avoid overhead irrigation'
        ],
        'prevention': 'Common in warm, wet conditions. Ensure proper plant spacing.',
        'urgency': 'Act within 2 weeks of first signs'
    },

    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': {
        'english_name': 'Tomato Yellow Leaf Curl Virus',
        'swahili_name': 'Virusi vya Manjano ya Nyanya',
        'severity': 'Critical',
        'description': 'Viral disease spread by whiteflies causing yellowing, curling leaves and severe stunting. No cure once infected.',
        'treatment': [
            'REMOVE infected plants immediately — there is no cure',
            'Control whitefly populations with insecticides (imidacloprid)',
            'Use yellow sticky traps to monitor and catch whiteflies',
            'Plant virus-resistant tomato varieties for next season',
            'Do not replant tomatoes in same area until whitefly population controlled'
        ],
        'prevention': 'Use whitefly-resistant tomato varieties. Install insect-proof nets in nurseries.',
        'urgency': 'REMOVE INFECTED PLANTS IMMEDIATELY to prevent spread'
    },

    'Tomato___Tomato_mosaic_virus': {
        'english_name': 'Tomato Mosaic Virus',
        'swahili_name': 'Virusi vya Mosaic ya Nyanya',
        'severity': 'High',
        'description': 'Viral disease causing mottled yellow-green mosaic pattern on leaves and distorted fruit. Spreads through contact.',
        'treatment': [
            'Remove and destroy infected plants — no chemical cure exists',
            'Wash hands thoroughly before handling plants',
            'Disinfect all tools with bleach solution (1:9 bleach to water)',
            'Control aphids which spread the virus',
            'Do not smoke near plants — tobacco carries a related virus'
        ],
        'prevention': 'Use certified virus-free seeds. Wash hands before entering field.',
        'urgency': 'Remove infected plants within 24 hours to prevent spread'
    },

    'Tomato___healthy': {
        'english_name': 'Healthy Tomato',
        'swahili_name': 'Nyanya Yenye Afya',
        'severity': 'None',
        'description': 'Your tomato crop appears healthy. No disease detected.',
        'treatment': ['Continue current farming practices', 'Monitor weekly for disease signs'],
        'prevention': 'Maintain proper spacing, balanced fertilization, and regular field monitoring.',
        'urgency': 'No action required'
    }
}

print(f"Disease information defined for {len(DISEASE_INFO)} classes")
print("\nSeverity breakdown:")
from collections import Counter
severities = Counter(v['severity'] for v in DISEASE_INFO.values())
for severity, count in sorted(severities.items()):
    print(f"  {severity}: {count} classes")

Disease information defined for 17 classes

Severity breakdown:
  Critical: 3 classes
  High: 4 classes
  Medium: 7 classes
  None: 3 classes


In [ ]:
import tensorflow as tf
import numpy as np
import os
import shutil

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 16  # Reduced from 32 to save RAM
EPOCHS = 10
IMAGES_PER_CLASS = 300  # Use only 300 images per class instead of ~1900

base = 'data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
train_path = os.path.join(base, 'train')
valid_path = os.path.join(base, 'valid')

kenya_crops = ['Corn_(maize)', 'Potato', 'Tomato']

# Create a smaller subset to avoid RAM crash
subset_path = 'data/kenya_subset/train'
subset_valid_path = 'data/kenya_subset/valid'

os.makedirs(subset_path, exist_ok=True)
os.makedirs(subset_valid_path, exist_ok=True)

kenya_classes = sorted([c for c in os.listdir(train_path)
                        if any(crop in c for crop in kenya_crops)])

print(f"Creating subset with {IMAGES_PER_CLASS} images per class...")

for cls in kenya_classes:
    # Train subset
    src = os.path.join(train_path, cls)
    dst = os.path.join(subset_path, cls)
    os.makedirs(dst, exist_ok=True)
    images = os.listdir(src)[:IMAGES_PER_CLASS]
    for img in images:
        shutil.copy(os.path.join(src, img), os.path.join(dst, img))

    # Validation subset
    src_val = os.path.join(valid_path, cls)
    dst_val = os.path.join(subset_valid_path, cls)
    os.makedirs(dst_val, exist_ok=True)
    if os.path.exists(src_val):
        val_images = os.listdir(src_val)[:100]
        for img in val_images:
            shutil.copy(os.path.join(src_val, img), os.path.join(dst_val, img))

print("Subset created")

# Load subset
train_ds = tf.keras.utils.image_dataset_from_directory(
    subset_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    subset_valid_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

class_names = train_ds.class_names
print(f"\nClasses: {len(class_names)}")
print(f"Training images: ~{len(class_names) * IMAGES_PER_CLASS}")

# Normalize — NO cache() to save RAM
AUTOTUNE = tf.data.AUTOTUNE
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(AUTOTUNE)
valid_ds = valid_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(AUTOTUNE)

print("Data pipeline ready")

Creating subset with 300 images per class...
Subset created
Found 5100 files belonging to 17 classes.
Found 1700 files belonging to 17 classes.

Classes: 17
Training images: ~5100
Data pipeline ready


In [ ]:
# Same architecture that worked well for Diabetic Retinopathy
# but now multi-class (17 classes) instead of binary
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,424,145 (9.25 MB)

 Trainable params: 166,161 (649.07 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=3,
            restore_best_weights=True
        )
    ]
)

print(f"\nBest validation accuracy: {max(history.history['val_accuracy']):.2%}")

Epoch 1/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 67s 140ms/step - accuracy: 0.3047 - loss: 2.2303 - val_accuracy: 0.6394 - val_loss: 1.3496
Epoch 2/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.5743 - loss: 1.3318 - val_accuracy: 0.7447 - val_loss: 0.9256
Epoch 3/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.6727 - loss: 1.0339 - val_accuracy: 0.7900 - val_loss: 0.7465
Epoch 4/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.7182 - loss: 0.8663 - val_accuracy: 0.8000 - val_loss: 0.6579
Epoch 5/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.7498 - loss: 0.7592 - val_accuracy: 0.8147 - val_loss: 0.5980
Epoch 6/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 20s 34ms/step - accuracy: 0.7722 - loss: 0.6833 - val_accuracy: 0.8329 - val_loss: 0.5528
Epoch 7/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.7955 - loss: 0.6278 - val_accuracy: 0.8482 - val_loss: 0.5232
Epoch 8/10
319/319 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.8163 - loss: 0.5698 - 

In [ ]:
# Save in native Keras format
model.save('crop_doctor_model.keras')
print("Model saved")

# Save class names — we need this exact order for predictions
import json
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print(f"Class names saved: {class_names}")

# Backup both to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('crop_doctor_model.keras', '/content/drive/MyDrive/crop_doctor_model.keras')
shutil.copy('class_names.json', '/content/drive/MyDrive/class_names.json')
print("Both files backed up to Google Drive")

# Check file size
import os
size = os.path.getsize('crop_doctor_model.keras') / (1024*1024)
print(f"Model size: {size:.1f} MB")

Model saved
Class names saved: ['Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
Mounted at /content/drive
Both files backed up to Google Drive
Model size: 11.1 MB
